# GEO tokens vs ViT accuracy: zero-shot pilot

Public data and models only (ImageNet-R, timm ViT-B/16). Each tokenizer cuts a 256x256 image into an exact token budget; tokens are shrunk to 16x16, embedded with the pretrained patch embedding, and placed at the mean of the grid positions they cover. The full-grid path is checked against the model's own forward first.

In [ ]:
# setup
import os, re, json, time, random
import numpy as np
import torch
import torch.nn.functional as F
import timm
from PIL import Image
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "timm", timm.__version__, "device", dev,
      torch.cuda.get_device_name(0) if dev == "cuda" else "")
N_IMAGES = 1000          # pilot; set to None for all of ImageNet-R
BUDGETS = (64, 100, 130, 160, 196, 256)
METHODS = ("geo_uniform", "geo_shrink", "pixel_blur", "entropy", "random")
SEED = 0

In [ ]:
# tokenizers (tokens_core.py)
"""Token layouts at an exact budget, for a 256x256 image with 16px leaves.

Every method gives each splittable node (256, 128, 64, 32 px) a "loss" score, then a
greedy quadtree splits the highest-loss node until the budget is reached. Each split
adds 3 tokens, so budgets are 1 + 3k (64, 100, 130, 160, 196, 256).

  geo_uniform : max pixel deviation from the node mean, YCbCr, chroma counted at half
                weight (canPrune's rule as a score: the q at which the node would merge)
  geo_shrink  : max pixel deviation from its own s/16 cell mean, same weighting
                (shrink-safe as a score: how much is lost shrinking the node to 16x16)
  pixel_blur  : Quadformer's baseline: sqrt(mean squared error of shrink-and-blow-up * s), RGB
  entropy     : 256-bin grey-level entropy (APT-style)
  random      : random score (sanity floor)
"""
import heapq
import numpy as np

IMG, LEAF = 256, 16
SIZES = (256, 128, 64, 32)          # splittable node sizes


def ycc(rgb01):
    r, g, b = rgb01[..., 0] * 255, rgb01[..., 1] * 255, rgb01[..., 2] * 255
    y = 0.299 * r + 0.587 * g + 0.114 * b
    cb = 128 - 0.168736 * r - 0.331264 * g + 0.5 * b
    cr = 128 + 0.5 * r - 0.418688 * g - 0.081312 * b
    return np.stack([y, cb, cr], -1)


def _blocks(a, s):
    """(H,W,C) -> (n, n, s, s, C) blocks of size s."""
    n = a.shape[0] // s
    return a.reshape(n, s, n, s, -1).transpose(0, 2, 1, 3, 4)


def _cell_mean(a, s):
    """Each pixel replaced by the mean of its s/16 cell inside s-blocks == shrink to 16 and blow up."""
    f = s // LEAF
    if f == 1:
        return a
    n = a.shape[0] // f
    m = a.reshape(n, f, n, f, -1).mean(axis=(1, 3))
    return np.repeat(np.repeat(m, f, 0), f, 1)


def scores(rgb01, method, rng=None):
    """dict size -> (n, n) loss array for every splittable node."""
    out = {}
    Y = ycc(rgb01)
    W = np.array([1.0, 0.5, 0.5])                       # chroma at half weight (canPrune: 2x tolerance)
    grey = np.clip(Y[..., 0], 0, 255).astype(np.int64)
    for s in SIZES:
        n = IMG // s
        if method == "geo_uniform":
            b = _blocks(Y, s)
            dev = np.abs(b - b.mean(axis=(2, 3), keepdims=True)) * W
            out[s] = dev.max(axis=(2, 3, 4))
        elif method == "geo_shrink":
            dev = np.abs(Y - _cell_mean(Y, s)) * W
            out[s] = _blocks(dev, s).max(axis=(2, 3, 4))
        elif method == "pixel_blur":
            se = (rgb01 - _cell_mean(rgb01, s)) ** 2
            mse = _blocks(se, s).mean(axis=(2, 3, 4))
            out[s] = np.sqrt(mse * s)
        elif method == "entropy":
            g = _blocks(grey[..., None], s)[..., 0]
            e = np.zeros((n, n))
            for i in range(n):
                for j in range(n):
                    h = np.bincount(g[i, j].ravel(), minlength=256).astype(float)
                    p = h[h > 0] / h.sum()
                    e[i, j] = -(p * np.log2(p)).sum()
            out[s] = e
        elif method == "random":
            out[s] = rng.random((n, n))
        else:
            raise ValueError(method)
    return out


def layout(sc, budget):
    """Greedy split: highest loss first. Returns list of (y, x, size)."""
    assert (budget - 1) % 3 == 0 and 1 <= budget <= (IMG // LEAF) ** 2, budget
    heap = [(-sc[IMG][0, 0], 0, 0, IMG)]
    leaves = []
    count = 1
    while count < budget:
        _, y, x, s = heapq.heappop(heap)
        h = s // 2
        for dy in (0, h):
            for dx in (0, h):
                yy, xx = y + dy, x + dx
                if h == LEAF:
                    leaves.append((yy, xx, h))
                else:
                    heapq.heappush(heap, (-sc[h][yy // h, xx // h], yy, xx, h))
        count += 3
        while heap and heap[0][3] == LEAF:      # never happens (leaves kept apart), safety
            leaves.append(heapq.heappop(heap)[1:])
    toks = leaves + [(y, x, s) for _, y, x, s in heap]
    assert len(toks) == budget and sum(s * s for _, _, s in toks) == IMG * IMG
    return toks


def grid(s):
    return [(y, x, s) for y in range(0, IMG, s) for x in range(0, IMG, s)]

In [ ]:
# data
roots = [r for r, d, f in os.walk("/kaggle/input") if re.fullmatch(r"n\d{8}", os.path.basename(r))]
data_root = os.path.dirname(roots[0])
wnid_dirs = sorted(d for d in os.listdir(data_root) if re.fullmatch(r"n\d{8}", d))
files = [(os.path.join(data_root, w, f), w) for w in wnid_dirs
         for f in sorted(os.listdir(os.path.join(data_root, w)))]
print(data_root, len(wnid_dirs), "classes", len(files), "images")
info = timm.data.ImageNetInfo()
in1k = list(info.label_names())
assert len(in1k) == 1000 and in1k[0].startswith("n"), in1k[:3]
sub_idx = torch.tensor([in1k.index(w) for w in wnid_dirs], device=dev)   # the 200 ImageNet-R classes
random.Random(SEED).shuffle(files)
if N_IMAGES:
    files = files[:N_IMAGES]

In [ ]:
# model
model = timm.create_model("vit_base_patch16_224.augreg2_in21k_ft_in1k", pretrained=True, img_size=256).to(dev).eval()
cfg = timm.data.resolve_model_data_config(model)
MEAN = torch.tensor(cfg["mean"], device=dev).view(3, 1, 1)
STD = torch.tensor(cfg["std"], device=dev).view(3, 1, 1)
assert getattr(model, "num_reg_tokens", 0) == 0 and not model.no_embed_class
POS = model.pos_embed[0, 1:].reshape(16, 16, -1).permute(2, 0, 1)        # (768, 16, 16) grid positions
POS_CLS = model.pos_embed[0, :1]


def load(path):
    im = Image.open(path).convert("RGB")
    w, h = im.size
    k = 256 / min(w, h)
    im = im.resize((max(256, round(w * k)), max(256, round(h * k))), Image.BICUBIC)
    w, h = im.size
    im = im.crop(((w - 256) // 2, (h - 256) // 2, (w - 256) // 2 + 256, (h - 256) // 2 + 256))
    return np.asarray(im, dtype=np.float64) / 255.0


@torch.no_grad()
def embed(img01_t, toks):
    """img01_t: (3,256,256) in [0,1] on device. toks: list of (y,x,s). -> (B, 768) token embeddings + pos."""
    x = (img01_t - MEAN) / STD
    pats, poss = [], []
    pooled = {}
    for y, xx, s in toks:
        f = s // 16
        if f not in pooled:
            pooled[f] = (F.avg_pool2d(x[None], f)[0] if f > 1 else x,
                         F.avg_pool2d(POS[None], f)[0] if f > 1 else POS)
        im_f, pos_f = pooled[f]
        i, j = y // f, xx // f
        pats.append(im_f[:, i:i + 16, j:j + 16])          # the token shrunk to 16x16 (area)
        poss.append(pos_f[:, y // s, xx // s])            # mean of the grid positions it covers
    p = model.patch_embed.proj(torch.stack(pats)).flatten(1)
    p = model.patch_embed.norm(p)
    return p + torch.stack(poss)


@torch.no_grad()
def classify(seqs):
    """seqs: (N, B, 768) token+pos. -> (N, 1000) logits."""
    cls = (model.cls_token[0] + POS_CLS).expand(seqs.shape[0], 1, -1)
    x = torch.cat([cls, seqs], 1)
    x = model.norm_pre(model.patch_drop(x))
    x = model.norm(model.blocks(x))
    return model.forward_head(x)

In [ ]:
# sanity - the full 16px grid through this path must equal the model's own forward
a = load(files[0][0])
t = torch.tensor(a, dtype=torch.float32, device=dev).permute(2, 0, 1)
ours = classify(embed(t, grid(16))[None])
ref = model(((t - MEAN) / STD)[None])
print("max |logit diff| full grid:", float((ours - ref).abs().max()))
assert torch.allclose(ours, ref, atol=1e-3), "token path does not match the model"

In [ ]:
# run
t0 = time.time()
CHUNK = 32
correct = {(m, b): 0 for m in METHODS for b in BUDGETS}
correct.update({("grid", 64): 0, ("grid", 256): 0})
n_done = 0
rng = np.random.default_rng(SEED)
for c0 in range(0, len(files), CHUNK):
    chunk = files[c0:c0 + CHUNK]
    imgs = [load(p) for p, _ in chunk]
    labels = torch.tensor([wnid_dirs.index(w) for _, w in chunk], device=dev)
    ts = [torch.tensor(a, dtype=torch.float32, device=dev).permute(2, 0, 1) for a in imgs]
    jobs = {("grid", 64): [grid(32)] * len(chunk), ("grid", 256): [grid(16)] * len(chunk)}
    for m in METHODS:
        scs = [scores(a, m, rng) for a in imgs]
        for b in BUDGETS:
            jobs[(m, b)] = [layout(sc, b) for sc in scs]
    for key, lays in jobs.items():
        seqs = torch.stack([embed(t, L) for t, L in zip(ts, lays)])
        pred = classify(seqs)[:, sub_idx].argmax(1)
        correct[key] += int((pred == labels).sum())
    n_done += len(chunk)
    if (c0 // CHUNK) % 5 == 0:
        print(f"{n_done}/{len(files)} images, {time.time() - t0:.0f}s, "
              f"full grid acc {correct[('grid', 256)] / n_done:.3f}", flush=True)

acc = {f"{m}|{b}": correct[(m, b)] / n_done for (m, b) in correct}
json.dump(dict(n=n_done, acc=acc, seconds=time.time() - t0), open("/kaggle/working/pilot_results.json", "w"), indent=1)
print(f"\nImageNet-R, {n_done} images, zero-shot ViT-B/16 @256, top-1 on the 200 classes")
print(f"{'method':<12}" + "".join(f"{b:>8}" for b in BUDGETS))
for m in METHODS:
    print(f"{m:<12}" + "".join(f"{100 * correct[(m, b)] / n_done:>7.1f}%" for b in BUDGETS))
print(f"{'grid':<12}{100 * correct[('grid', 64)] / n_done:>7.1f}% (64 tokens, 32px patches)   "
      f"{100 * correct[('grid', 256)] / n_done:.1f}% (256 tokens, full)")

In [ ]:
# plot
import matplotlib.pyplot as plt
COL = {"geo_uniform": "#c2185b", "geo_shrink": "#ef6c00", "pixel_blur": "#6a1b9a", "entropy": "#1565c0", "random": "#9e9e9e"}
fig, ax = plt.subplots(figsize=(7, 5))
for m in METHODS:
    ax.plot(BUDGETS, [100 * correct[(m, b)] / n_done for b in BUDGETS], "o-", color=COL[m], label=m)
ax.plot([64], [100 * correct[("grid", 64)] / n_done], "ks", label="grid 32px (64 tokens)")
ax.axhline(100 * correct[("grid", 256)] / n_done, color="k", lw=0.8, ls=":", label="full grid (256 tokens)")
ax.set_xlabel("tokens"); ax.set_ylabel("top-1 accuracy, %"); ax.legend(fontsize=8)
ax.set_title(f"ImageNet-R zero-shot, ViT-B/16 @256, n={n_done}")
fig.tight_layout(); fig.savefig("/kaggle/working/pilot_accuracy.png", dpi=110)